# UMAP Diagnostic — Interactive, Site-D Focused (6 drugs)

Interactive companion to `UMAP-Diagnostic.ipynb`. Same data pipeline
(log1p + standardize, dedup by isolate, site-stratified subsampling) but:

- **Hover provenance**: every point exposes `code`, `species`, `site`,
  `cluster`, drug coverage (`n/6 drugs R`) — and the resistance label for
  per-drug views.
- **Toggle visibility**: click legend entries (native Plotly), or use the
  two dropdowns:
  - **Color by**: Site / Species / Resistance / Cluster
  - **Highlight**: All / D vs rest / each site only / D-dominant clusters
    (opacity-dims everything else)
- **Cluster pinpointing**: clusters are computed in PCA space (not on the 2D
  UMAP), ranked by **D-lift** (%D in cluster / %D overall) and saved together
  with species/resistance composition.
- **Selection**: in local JupyterLab the figure is a `FigureWidget` — lasso or
  box-select points to get a live provenance table. In Colab the same
  information is available via hover + the dropdowns.
- **Exports**: standalone HTML per view + composition/separability CSVs +
  static PDF snapshots.

**Site D focus**: D-vs-rest coloring, D-only embedding, D-lift cluster
ranking, and one-vs-rest separability metrics (silhouette, 5-NN AUC, linear
probe AUC) per site — before/after species masking when masks are available.

`QUICK_TEST=1` (`os.environ`) runs a small smoke version; caches and outputs
are isolated under `results_quick/`.

## 1. Setup

In [ ]:
# ── CONFIG ──
import os

ALL_DRUGS = [
    "Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic_acid",
    "Piperacillin-Tazobactam", "Ceftriaxone", "Ceftazidime",
]
SITE_ORDER = ["A", "B", "C", "D"]

MASK_TOP_K = 500

# UMAP parameters
UMAP_N_NEIGHBORS = 20
UMAP_MIN_DIST = 0.25
UMAP_N_COMPONENTS = 2
UMAP_RANDOM_STATE = 42

# Subsampling (site-stratified) to keep UMAP tractable + readable
GLOBAL_SUBSAMPLE = 20000
PER_DRUG_SUBSAMPLE = 8000
D_ONLY_SUBSAMPLE = 8000

# Clustering (in PCA space, not on the UMAP embedding)
PCA_COMPONENTS = 50
HDBSCAN_MIN_CLUSTER = 200

# Interactive figures
SPECIES_TOP = 12
HTML_PLOTLYJS = "cdn"           # "cdn" (small, needs internet) or True (offline, +3.5 MB/file)
DRUG_TO_SHOW = "Ceftriaxone"    # displayed inline; all drugs exported to HTML

RUN_NAME = "UMAP-Diagnostic-Interactive"

# QUICK_TEST=1 -> small smoke run (2 drugs, tiny subsamples); caches/outputs isolated
QUICK_TEST = os.environ.get("UMAP_DIAG_QUICK", "0") == "1"
if QUICK_TEST:
    ALL_DRUGS = ALL_DRUGS[:2]
    GLOBAL_SUBSAMPLE = 3000
    PER_DRUG_SUBSAMPLE = 2000
    D_ONLY_SUBSAMPLE = 2000
    HDBSCAN_MIN_CLUSTER = 60
    print("[QUICK_TEST] reduced scope (smoke run)")
else:
    print("[FULL] 6 drugs, global=20k, per-drug=8k, D-only=8k")

In [ ]:
!pip install plotly umap-learn anywidget --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("IN_COLAB =", IN_COLAB)

In [ ]:
import gc
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

import umap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import HDBSCAN, MiniBatchKMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
print(f"plotly {plotly.__version__} | umap {umap.__version__}")

In [ ]:
# ── Paths (Colab + local auto-detect) ──
if IN_COLAB:
    BASE = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    here = Path.cwd()
    candidates = [
        Path("/media/asd/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed"),
        Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed"),
    ]
    for up in (3, 2, 4):
        if len(here.parents) > up:
            candidates.append(here.parents[up])
    BASE = next((p for p in candidates if (p / "Proc_DRIAMS-A").exists()), candidates[0])

_ANALYSIS_CANDIDATES = [
    Path("/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Analysis"),
    BASE / "Processing" / "Analysis",
]
ANALYSIS_DIR = next((p for p in _ANALYSIS_CANDIDATES if (p / "08-Federated-mlp-lr-rf").exists()), _ANALYSIS_CANDIDATES[1])
SPECIES_MASK_DIR = ANALYSIS_DIR / "08-Federated-mlp-lr-rf" / "Species-Masking"
SHARED_MASKS_DIR = SPECIES_MASK_DIR / "shared_masks"
FL_DIR = ANALYSIS_DIR / "08-Federated-mlp-lr-rf" / "Drugs"

RUN_TAG = "quick" if QUICK_TEST else "full"
OUT_DIR = SPECIES_MASK_DIR / RUN_NAME / (f"results_{RUN_TAG}" if QUICK_TEST else "results")
EMB_DIR = OUT_DIR / "embeddings"
for p in (OUT_DIR, EMB_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("BASE:  ", BASE)
print("masks: ", SHARED_MASKS_DIR, "(exists)" if SHARED_MASKS_DIR.exists() else "(missing)")
print("output:", OUT_DIR)

## 2. Load data + build provenance

In [ ]:
def load_drug(drug):
    codes, Xs, ys, sps, sites = [], [], [], [], []
    for si, site in enumerate(SITE_ORDER):
        path = BASE / f"Proc_DRIAMS-{site}" / drug / "data.csv"
        df = pd.read_csv(path)
        bin_cols = [c for c in df.columns if c.startswith("bin_")]
        codes.append(df["code"].values)
        Xs.append(df[bin_cols].to_numpy(dtype="float32"))
        ys.append(df["label"].to_numpy(dtype="int64"))
        sps.append(df["species"].values)
        sites.append(np.full(len(df), si, dtype=int))
    return (np.concatenate(codes), np.concatenate(Xs, axis=0),
            np.concatenate(ys), np.concatenate(sps), np.concatenate(sites))


raw = {}
for drug in ALL_DRUGS:
    raw[drug] = load_drug(drug)
    print(f"  {drug}: {len(raw[drug][1]):,} spectra")
print(f"Total spectra (pre-dedup): {sum(len(raw[d][1]) for d in ALL_DRUGS):,}")

In [ ]:
def preprocess(X):
    Xl = np.log1p(X.astype("float32"))
    return StandardScaler().fit_transform(Xl)

print("preprocess() ready (log1p + standardize)")

In [ ]:
# ── Global pooled dataset (dedup by isolate code) + per-isolate provenance ──
code_all = np.concatenate([raw[d][0] for d in ALL_DRUGS])
X_all    = np.concatenate([raw[d][1] for d in ALL_DRUGS], axis=0)
sp_all   = np.concatenate([raw[d][3] for d in ALL_DRUGS])
site_all = np.concatenate([raw[d][4] for d in ALL_DRUGS])
drug_all = np.concatenate([np.full(len(raw[d][1]), d) for d in ALL_DRUGS])

# How many of the 6 drugs has each isolate been tested against / found resistant to?
cov_n, cov_r = defaultdict(int), defaultdict(int)
for drug in ALL_DRUGS:
    for c, lab, si in zip(raw[drug][0], raw[drug][2], raw[drug][4]):
        key = (int(si), c)
        cov_n[key] += 1
        cov_r[key] += int(lab)

_, first_idx = np.unique(code_all, return_index=True)
first_idx.sort()
X_g_raw = X_all[first_idx]
code_g, species_g, site_g = code_all[first_idx], sp_all[first_idx], site_all[first_idx]
drug_g = drug_all[first_idx]
n_tested_g = np.array([cov_n[(int(si), c)] for si, c in zip(site_g, code_g)])
n_res_g = np.array([cov_r[(int(si), c)] for si, c in zip(site_g, code_g)])

meta_g = pd.DataFrame({
    "row_id": np.arange(len(code_g)),
    "code": code_g,
    "species": species_g,
    "site": [SITE_ORDER[i] for i in site_g],
    "site_idx": site_g,
    "first_drug": drug_g,
    "n_drugs_tested": n_tested_g,
    "n_resistant": n_res_g,
    "frac_resistant": n_res_g / np.maximum(1, n_tested_g),
})

print(f"Global: {len(code_all):,} spectra -> {len(first_idx):,} unique isolates")
print(meta_g["site"].value_counts().to_string())

del X_all
gc.collect()

In [ ]:
X_g = preprocess(X_g_raw)
del X_g_raw
gc.collect()
print("Global preprocessed:", X_g.shape)

## 3. Helpers: subsampling, UMAP, masks, clustering, composition

In [ ]:
def subsample_idx(labels, n, seed=SEED):
    labels = np.asarray(labels)
    n = int(min(n, len(labels)))
    if n == len(labels):
        return np.arange(len(labels))
    rng = np.random.default_rng(seed)
    chosen = []
    for u in np.unique(labels):
        gi = np.where(labels == u)[0]
        k = min(max(1, int(round(n * len(gi) / len(labels)))), len(gi))
        chosen.append(rng.choice(gi, size=k, replace=False))
    sel = np.concatenate(chosen)
    if len(sel) > n:
        sel = rng.choice(sel, size=n, replace=False)
    return np.sort(sel)


def fit_umap(X):
    reducer = umap.UMAP(n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                        n_components=UMAP_N_COMPONENTS, random_state=UMAP_RANDOM_STATE,
                        metric="euclidean")
    return reducer.fit_transform(X)


def cached_embedding(name, X, idx):
    path = EMB_DIR / f"{name}.npz"
    if path.exists():
        z = np.load(path)
        if len(z["idx"]) == len(idx) and np.array_equal(z["idx"], idx):
            print(f"  [cache] {name}: {z['emb'].shape}")
            return z["emb"], z["idx"]
    emb = fit_umap(X[idx])
    np.savez_compressed(path, emb=emb, idx=idx)
    print(f"  [fit]   {name}: {emb.shape}")
    return emb, idx


def apply_mask(X, mask):
    Xc = X.copy()
    if mask is not None and len(mask) > 0:
        Xc[:, np.asarray(mask, dtype=int)] = 0.0
    return Xc


def load_mask(name):
    p = SHARED_MASKS_DIR / name
    if p.exists():
        return np.load(p).astype(int)
    return np.array([], dtype=int)

In [ ]:
def cluster_embedding(X, min_cluster_size=HDBSCAN_MIN_CLUSTER, n_pca=PCA_COMPONENTS):
    # Cluster in PCA space (not on the 2D UMAP embedding); HDBSCAN with KMeans fallback
    n_comp = int(min(n_pca, X.shape[1], X.shape[0] - 1))
    Xp = PCA(n_components=n_comp, random_state=SEED).fit_transform(X)
    n_mc = int(min(min_cluster_size, max(20, X.shape[0] // 10)))
    lab, frac_noise = None, 1.0
    try:
        lab = HDBSCAN(min_cluster_size=n_mc, min_samples=max(5, n_mc // 10)).fit_predict(Xp)
        frac_noise = float((lab == -1).mean())
        n_clusters = len(set(lab[lab >= 0]))
    except Exception as e:
        print(f"  HDBSCAN failed ({e}) -> KMeans fallback")
        n_clusters = 0
    if lab is None or frac_noise > 0.5 or n_clusters < 2:
        step = max(1, len(Xp) // 2500)
        sub = Xp[::step]
        best_lab, best_s, best_k = None, -1.0, 8
        for k in [4, 6, 8, 10, 12]:
            if k >= len(Xp):
                continue
            lab_k = MiniBatchKMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(Xp)
            s = silhouette_score(sub, lab_k[::step])
            if s > best_s:
                best_lab, best_s, best_k = lab_k, s, k
        lab = best_lab
        print(f"  KMeans fallback: k={best_k}, silhouette={best_s:.3f} (HDBSCAN noise={frac_noise:.0%})")
    else:
        print(f"  HDBSCAN: {n_clusters} clusters, noise={frac_noise:.0%}")
    return lab, Xp


def cluster_composition(meta, labels, top_species=5, resistance_col=None):
    # Cluster x site/species/resistance composition, ranked by D-lift
    df = meta.copy().reset_index(drop=True)
    df["cluster"] = labels
    site_share_all = df["site"].value_counts(normalize=True)
    d_all = float(site_share_all.get("D", 0.0))
    rows = []
    for c, g in df.groupby("cluster", sort=True):
        share = g["site"].value_counts(normalize=True)
        if resistance_col and resistance_col in g.columns:
            r_rate = float(g[resistance_col].mean())
        elif "n_drugs_tested" in g.columns:
            r_rate = float(g["n_resistant"].sum() / max(1, g["n_drugs_tested"].sum()))
        else:
            r_rate = np.nan
        sp = g["species"].value_counts().head(top_species)
        rows.append({
            "cluster": int(c),
            "n": int(len(g)),
            "pct_D": float(share.get("D", 0.0)),
            "D_lift": float(share.get("D", 0.0) / d_all) if d_all else np.nan,
            "pct_A": float(share.get("A", 0.0)),
            "pct_B": float(share.get("B", 0.0)),
            "pct_C": float(share.get("C", 0.0)),
            "resistance_rate": r_rate,
            "top_species": "; ".join(f"{s} ({v / len(g):.0%})" for s, v in sp.items()),
        })
    out = pd.DataFrame(rows).sort_values("D_lift", ascending=False).reset_index(drop=True)
    return out

## 4. Global embedding + clustering (ranked by D-lift)

In [ ]:
gs = subsample_idx(site_g, GLOBAL_SUBSAMPLE)
emb_g, gs = cached_embedding("global_umap", X_g, gs)

meta_gs = meta_g.iloc[gs].reset_index(drop=True).copy()
lab_g, Xp_g = cluster_embedding(X_g[gs])
meta_gs["cluster"] = lab_g
meta_gs["umap1"], meta_gs["umap2"] = emb_g[:, 0], emb_g[:, 1]

comp_g = cluster_composition(meta_gs, lab_g, resistance_col=None)
comp_g.to_csv(OUT_DIR / "cluster_composition_global.csv", index=False)
print("\nClusters ranked by D-lift (global):")
display(comp_g.head(12))

## 5. Interactive figure builder

One reusable builder produces all interactive figures. Each figure contains
traces for every color mode, switched by the **Color by** dropdown; the
**Highlight** dropdown applies per-point opacity (D-vs-rest, single site, or
D-dominant clusters). Hover exposes full provenance; legend clicks toggle
site/species/cluster visibility.

In [ ]:
SITE_COLORS = {"A": "#1f77b4", "B": "#2ca02c", "C": "#ff7f0e", "D": "#d62728"}
CLUSTER_PALETTE = px.colors.qualitative.Dark24


def _prepare_common(meta):
    df = meta.reset_index(drop=True).copy()
    df["row_id"] = np.arange(len(df))  # positional id within this figure's meta
    fields = ["row_id", "code", "species", "site"]
    if "drug" in df.columns:
        fields.append("drug")
    if "first_drug" in df.columns:
        fields.append("first_drug")
    if "cluster" in df.columns:
        fields.append("cluster")
    if "label" in df.columns:
        df["res_label"] = df["label"].map({0: "S", 1: "R"})
        fields.append("res_label")
    if "n_drugs_tested" in df.columns:
        df["coverage"] = (df["n_resistant"].astype(str) + "/" +
                          df["n_drugs_tested"].astype(str) + " drugs R")
        df["frac_R"] = df["frac_resistant"].map(lambda v: f"{v:.2f}")
        fields += ["coverage", "frac_R"]
    cd = np.column_stack([df[f].astype(str).values for f in fields])
    return df, fields, cd


def build_interactive_fig(meta, emb, title, top_species=SPECIES_TOP, n_highlight_clusters=8):
    df, fields, cd = _prepare_common(meta)
    n = len(df)
    hover = "<br>".join(f"{f}: %{{customdata[{i}]}}"
                        for i, f in enumerate(fields) if f != "row_id")
    hover += "<extra>%{fullData.name}</extra>"

    traces, mode_traces = [], {}

    def add(mask, name, color, visible=False, color_arr=None):
        idx = np.where(mask)[0]
        tr = go.Scattergl(
            x=emb[idx, 0], y=emb[idx, 1], mode="markers", name=name,
            marker=dict(size=4, opacity=0.75,
                        color=(color if color_arr is None else color_arr[idx])),
            customdata=cd[idx], hovertemplate=hover, visible=visible,
            selected=dict(marker=dict(size=7, opacity=1.0)),
            unselected=dict(marker=dict(opacity=0.10)),
        )
        traces.append(tr)
        return len(traces) - 1

    # Color by site (default visible)
    mode_traces["Site"] = [add(df["site"].values == s, f"Site {s}", SITE_COLORS[s], visible=True)
                           for s in SITE_ORDER]

    # Color by species (top-N + other)
    top_sp = df["species"].value_counts().head(top_species).index.tolist()
    pal = px.colors.qualitative.Alphabet
    is_top = df["species"].isin(top_sp).values
    mode_traces["Species"] = [add(df["species"].values == sp, sp, pal[i % len(pal)])
                              for i, sp in enumerate(top_sp)]
    if (~is_top).any():
        mode_traces["Species"].append(add(~is_top, "other", "#bdbdbd"))

    # Color by resistance
    if "label" in df.columns:
        mode_traces["Resistance"] = [
            add(df["label"].values == 0, "Susceptible", "#1f77b4"),
            add(df["label"].values == 1, "Resistant", "#d62728"),
        ]
    else:
        tr = go.Scattergl(
            x=emb[:, 0], y=emb[:, 1], mode="markers", name="R fraction",
            marker=dict(size=4, opacity=0.85, color=df["frac_resistant"].astype(float),
                        colorscale="RdYlBu_r", showscale=True,
                        colorbar=dict(title="R fraction", thickness=12)),
            customdata=cd, hovertemplate=hover, visible=False,
            selected=dict(marker=dict(size=7, opacity=1.0)),
            unselected=dict(marker=dict(opacity=0.10)),
        )
        traces.append(tr)
        mode_traces["Resistance"] = [len(traces) - 1]

    # Color by cluster
    if "cluster" in df.columns:
        cl = df["cluster"].values
        mode_traces["Cluster"] = []
        for i, c in enumerate(sorted([c for c in np.unique(cl) if c != -1])):
            mode_traces["Cluster"].append(add(cl == c, f"Cluster {int(c)}",
                                              CLUSTER_PALETTE[i % len(CLUSTER_PALETTE)]))
        if (cl == -1).any():
            mode_traces["Cluster"].append(add(cl == -1, "noise", "#bdbdbd"))

    # Color-by dropdown
    mode_buttons = []
    for mode, idxs in mode_traces.items():
        vis = [False] * len(traces)
        for i in idxs:
            vis[i] = True
        mode_buttons.append(dict(label=mode, method="restyle", args=[{"visible": vis}]))

    # Highlight dropdown: per-point opacity
    opts = {"All": np.full(n, 0.75)}
    for s in SITE_ORDER:
        opts["D only" if s == "D" else f"{s} only"] = np.where(df["site"].values == s, 0.95, 0.02)
    opts["D vs rest"] = np.where(df["site"].values == "D", 0.95, 0.05)
    if "cluster" in df.columns:
        pct_d = df.groupby("cluster")["site"].apply(lambda s: (s == "D").mean())
        for c in pct_d.sort_values(ascending=False).head(n_highlight_clusters).index:
            if c == -1:
                continue
            opts[f"Cluster {int(c)} ({pct_d[c]:.0%} D)"] = np.where(df["cluster"].values == c, 0.95, 0.03)

    hl_buttons = []
    for label, op in opts.items():
        per_trace = [op[np.asarray(tr.customdata)[:, 0].astype(int)] for tr in traces]
        hl_buttons.append(dict(label=label, method="restyle", args=[{"marker.opacity": per_trace}]))

    if "cluster" in df.columns:
        sub = f"{n:,} isolates · {len(set(meta['cluster']))} clusters"
    else:
        sub = f"{n:,} isolates"
    dd_kw = dict(direction="down", showactive=True, yanchor="top", y=1.13,
                 bgcolor="rgba(255,255,255,0.9)")
    fig = go.Figure(data=traces, layout=go.Layout(
        title=dict(text=f"{title}<br><sup>{sub}</sup>", x=0.0),
        height=700, template="plotly_white", hovermode="closest",
        dragmode="select", uirevision="umap",
        legend=dict(itemsizing="constant", font=dict(size=9), groupclick="toggleitem"),
        margin=dict(t=150, r=10, b=40),
        updatemenus=[
            dict(buttons=mode_buttons, x=0.0, xanchor="left", **dd_kw),
            dict(buttons=hl_buttons, x=0.26, xanchor="left", **dd_kw),
        ],
        annotations=[
            dict(text="Color by:", x=0.0, xref="paper", y=1.16, yref="paper",
                 showarrow=False, font=dict(size=11)),
            dict(text="Highlight:", x=0.26, xref="paper", y=1.16, yref="paper",
                 showarrow=False, font=dict(size=11)),
        ],
        xaxis=dict(title="UMAP1", showgrid=False, zeroline=False),
        yaxis=dict(title="UMAP2", showgrid=False, zeroline=False),
    ))
    return fig

In [ ]:
def show_figure(fig, meta):
    # Show interactive figure; local JupyterLab adds lasso/box selection -> provenance table
    if IN_COLAB:
        fig.show()
        return
    try:
        import ipywidgets as widgets

        fw = go.FigureWidget(fig)
        out = widgets.Output()
        base_cols = [c for c in ["site", "species", "cluster", "label", "drug", "first_drug",
                                 "n_drugs_tested", "n_resistant", "code"] if c in meta.columns]

        def on_select(trace, points, selector):
            inds = np.asarray(points.point_inds, dtype=int)
            if len(inds) == 0:
                return
            row_ids = np.asarray(trace.customdata)[inds][:, 0].astype(int)
            sub = meta.iloc[row_ids]
            with out:
                out.clear_output(wait=True)
                print(f"Selected {len(sub):,} isolates")
                display(sub[base_cols].head(200))
                if "site" in sub:
                    print("site composition:")
                    print(sub["site"].value_counts(normalize=True).round(3).to_string())
                if "cluster" in sub:
                    print("cluster composition:")
                    print(sub["cluster"].value_counts().head(10).to_string())
                if "label" in sub:
                    print(f"resistance rate: {sub['label'].mean():.3f}")
                elif "n_drugs_tested" in sub:
                    print(f"resistance rate: {sub['n_resistant'].sum() / max(1, sub['n_drugs_tested'].sum()):.3f}")

        for tr in fw.data:
            tr.on_selection(on_select)
        display(fw)
        print("Lasso/box-select points above to see their provenance below.")
        display(out)
    except Exception as e:
        print(f"FigureWidget unavailable ({e}); falling back to standard renderer")
        fig.show()

In [ ]:
fig_global = build_interactive_fig(meta_gs, emb_g, "Global UMAP — 6 drugs, dedup by isolate")
html_path = OUT_DIR / "umap_global_interactive.html"
fig_global.write_html(html_path, include_plotlyjs=HTML_PLOTLYJS, full_html=True)
print("saved", html_path)
show_figure(fig_global, meta_gs)

## 6. Per-drug embeddings + clusters

In [ ]:
per_drug_meta = {}
for drug in ALL_DRUGS:
    code_d, X_d, y_d, sp_d, site_d = raw[drug]
    Xd = preprocess(X_d)
    ds = subsample_idx(site_d, PER_DRUG_SUBSAMPLE)
    emb_d, ds = cached_embedding(f"perdrug_{drug}_umap", Xd, ds)

    meta_d = pd.DataFrame({
        "row_id": np.arange(len(code_d)),
        "code": code_d,
        "species": sp_d,
        "site": [SITE_ORDER[i] for i in site_d],
        "site_idx": site_d,
        "label": y_d,
        "drug": drug,
    }).iloc[ds].reset_index(drop=True)
    lab_d, _ = cluster_embedding(Xd[ds])
    meta_d["cluster"] = lab_d
    meta_d["umap1"], meta_d["umap2"] = emb_d[:, 0], emb_d[:, 1]

    comp_d = cluster_composition(meta_d, lab_d, resistance_col="label")
    comp_d.to_csv(OUT_DIR / f"cluster_composition_{drug}.csv", index=False)

    fig_d = build_interactive_fig(meta_d, emb_d,
                                  f"{drug} — site / species / resistance / cluster")
    fig_d.write_html(OUT_DIR / f"umap_{drug}_interactive.html",
                     include_plotlyjs=HTML_PLOTLYJS, full_html=True)
    per_drug_meta[drug] = meta_d
    del Xd
    gc.collect()
    print(f"  {drug}: {len(meta_d):,} isolates, {comp_d.shape[0]} clusters, top D-lift "
          f"cluster {comp_d.iloc[0]['cluster']} ({comp_d.iloc[0]['D_lift']:.2f}x)")

print("\nSaved per-drug HTML:", ", ".join(f"umap_{d}_interactive.html" for d in ALL_DRUGS))
if DRUG_TO_SHOW in per_drug_meta:
    mt = per_drug_meta[DRUG_TO_SHOW]
    fig_show = build_interactive_fig(mt, mt[["umap1", "umap2"]].to_numpy(),
                                     f"{DRUG_TO_SHOW} — interactive")
    show_figure(fig_show, mt)

## 7. Site D only — internal structure

In [ ]:
d_idx = SITE_ORDER.index("D")
d_rows = np.where(site_g == d_idx)[0]
if len(d_rows) > D_ONLY_SUBSAMPLE:
    rng = np.random.default_rng(SEED)
    d_rows = np.sort(rng.choice(d_rows, size=D_ONLY_SUBSAMPLE, replace=False))
emb_donly, d_rows = cached_embedding("siteD_only_umap", X_g, d_rows)

meta_donly = meta_g.iloc[d_rows].reset_index(drop=True).copy()
lab_donly, _ = cluster_embedding(X_g[d_rows], min_cluster_size=max(40, HDBSCAN_MIN_CLUSTER // 2))
meta_donly["cluster"] = lab_donly
meta_donly["umap1"], meta_donly["umap2"] = emb_donly[:, 0], emb_donly[:, 1]

comp_donly = cluster_composition(meta_donly, lab_donly, resistance_col=None)
comp_donly.to_csv(OUT_DIR / "cluster_composition_siteD.csv", index=False)
fig_donly = build_interactive_fig(meta_donly, emb_donly,
                                  "Site D only — species / drug coverage / resistance / cluster")
fig_donly.write_html(OUT_DIR / "umap_siteD_interactive.html",
                     include_plotlyjs=HTML_PLOTLYJS, full_html=True)
print("Site D clusters (top D-lift):")
display(comp_donly.head(10))
show_figure(fig_donly, meta_donly)

## 8. Static snapshot (PDF) for docs

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
d_mask = meta_gs["site"].values == "D"
axes[0].scatter(emb_g[~d_mask, 0], emb_g[~d_mask, 1], s=3, c="#cccccc", alpha=0.5, label="A/B/C")
axes[0].scatter(emb_g[d_mask, 0], emb_g[d_mask, 1], s=3, c="#d62728", alpha=0.8, label="D")
axes[0].set_title("Global UMAP — site D vs rest")
axes[0].legend(markerscale=4)
axes[0].set_xlabel("UMAP1"); axes[0].set_ylabel("UMAP2")

hm = comp_g.set_index("cluster")[["pct_A", "pct_B", "pct_C", "pct_D"]].T
im = axes[1].imshow(hm.values, aspect="auto", cmap="magma")
axes[1].set_xticks(range(hm.shape[1])); axes[1].set_xticklabels(hm.columns, fontsize=7)
axes[1].set_yticks(range(4)); axes[1].set_yticklabels(["A", "B", "C", "D"])
axes[1].set_title("Cluster x site share")
plt.colorbar(im, ax=axes[1], fraction=0.046)

top = comp_g.head(12)
axes[2].barh(top["cluster"].astype(str)[::-1], top["D_lift"][::-1], color="#d62728")
axes[2].axvline(1.0, color="k", lw=1, ls="--")
axes[2].set_title("Clusters ranked by D-lift (top 12)")
axes[2].set_xlabel("D share / overall D share")

plt.tight_layout()
plt.savefig(OUT_DIR / "siteD_cluster_overview.pdf", bbox_inches="tight")
plt.show()

## 9. Site separability — is D the most distinct? (before/after masking)

In [ ]:
union_mask = load_mask("union_mask.npy")
majority_mask = load_mask("majority_mask.npy")
persite_masks = {s: load_mask(f"persite_site_{s}_mask.npy") for s in SITE_ORDER}

mask_variants = [("unmasked", None)]
if len(majority_mask):
    mask_variants.append(("majority", majority_mask))
if len(union_mask):
    mask_variants.append(("union", union_mask))
print("mask variants:", [m for m, _ in mask_variants],
      "| sizes:", {m: (0 if v is None else len(v)) for m, v in mask_variants})


def one_vs_rest_metrics(X, site_arr, n_sil=2500, n_probe=5000):
    rows = []
    for s in SITE_ORDER:
        y = (site_arr == SITE_ORDER.index(s)).astype(int)
        sel = subsample_idx(y, n_sil)
        sil = silhouette_score(X[sel], y[sel])
        sel2 = subsample_idx(y, n_probe)
        Xp, yp = X[sel2], y[sel2]
        knn_acc = cross_val_score(KNeighborsClassifier(5), Xp, yp, cv=5,
                                  scoring="accuracy", n_jobs=-1).mean()
        knn_auc = cross_val_score(KNeighborsClassifier(5), Xp, yp, cv=5,
                                  scoring="roc_auc", n_jobs=-1).mean()
        lin_auc = cross_val_score(SGDClassifier(loss="log_loss", max_iter=1000, random_state=SEED),
                                  Xp, yp, cv=5, scoring="roc_auc", n_jobs=-1).mean()
        rows.append({"site": s, "n": int(y.sum()), "silhouette": sil, "knn5_acc": knn_acc,
                     "knn5_auc": knn_auc, "linear_auc": lin_auc})
    return pd.DataFrame(rows)


sep_rows = []
for name, mask in mask_variants:
    Xv = X_g if mask is None else apply_mask(X_g, mask)
    dfm = one_vs_rest_metrics(Xv, site_g)
    dfm["mask"] = name
    sep_rows.append(dfm)
    print(f"  {name} done")
sep_df = pd.concat(sep_rows, ignore_index=True)
sep_df.to_csv(OUT_DIR / "site_separability.csv", index=False)
display(sep_df.pivot(index="site", columns="mask",
                     values=["knn5_auc", "linear_auc", "silhouette"]).round(3))

In [ ]:
metrics = [("silhouette", "Silhouette (one-vs-rest)"),
           ("knn5_auc", "5-NN AUC"),
           ("linear_auc", "Linear probe AUC")]
fig_sep = make_subplots(rows=1, cols=3, subplot_titles=[m[1] for m in metrics])
for ci, (metric, _) in enumerate(metrics, start=1):
    for m in sep_df["mask"].unique():
        sub = sep_df[sep_df["mask"] == m].set_index("site").reindex(SITE_ORDER)
        fig_sep.add_trace(
            go.Bar(x=SITE_ORDER, y=sub[metric], name=m, legendgroup=m, showlegend=(ci == 1)),
            row=1, col=ci)
fig_sep.update_layout(barmode="group", height=450, template="plotly_white",
                      title="Site separability (higher = more distinct; D expected highest)")
fig_sep.write_html(OUT_DIR / "site_separability.html",
                   include_plotlyjs=HTML_PLOTLYJS, full_html=True)
fig_sep.show()

## 10. Per-drug D separability + link to FL performance

In [ ]:
d_idx = SITE_ORDER.index("D")
dsep_rows = []
for drug in ALL_DRUGS:
    _, X_d, _, _, site_d = raw[drug]
    Xd = preprocess(X_d)
    y = (site_d == d_idx).astype(int)
    sel = subsample_idx(y, min(3000, len(y)))
    Xp, yp = Xd[sel], y[sel]
    sil = silhouette_score(Xp, yp)
    knn_auc = cross_val_score(KNeighborsClassifier(5), Xp, yp, cv=3,
                              scoring="roc_auc", n_jobs=-1).mean()
    lin_auc = cross_val_score(SGDClassifier(loss="log_loss", max_iter=1000, random_state=SEED),
                              Xp, yp, cv=3, scoring="roc_auc", n_jobs=-1).mean()
    dsep_rows.append({"drug": drug, "n_D": int(y.sum()), "silhouette": sil,
                      "knn5_auc": knn_auc, "linear_auc": lin_auc})
    del Xd
    gc.collect()
dsep_df = pd.DataFrame(dsep_rows)
dsep_df.to_csv(OUT_DIR / "siteD_separability_per_drug.csv", index=False)
display(dsep_df.round(3))

# Optional join with the latest FL run per drug (D_BalAcc)
fl_rows = []
if FL_DIR.exists():
    for drug in ALL_DRUGS:
        ddir = FL_DIR / drug
        cands = sorted([p for p in ddir.glob("[0-9]*-Run")
                        if (p / "results" / "final_results.csv").exists()])
        if not cands and (ddir / "results" / "final_results.csv").exists():
            cands = [ddir]
        if not cands:
            continue
        fr = pd.read_csv(cands[-1] / "results" / "final_results.csv")
        row = fr[fr["Method"].astype(str).str.startswith("FL FedAvg (MLP)")]
        if len(row):
            fl_rows.append({"drug": drug, "run": cands[-1].name,
                            "D_BalAcc": float(row.iloc[0]["D_BalAcc"]),
                            "All_BalAcc": float(row.iloc[0]["All_BalAcc"])})
fl_df = pd.DataFrame(fl_rows)
if len(fl_df):
    j = dsep_df.merge(fl_df, on="drug")
    j.to_csv(OUT_DIR / "siteD_separability_vs_FL.csv", index=False)
    fig_j = px.scatter(j, x="linear_auc", y="D_BalAcc", text="drug", size="n_D",
                       hover_data=["run", "All_BalAcc"],
                       title="Site-D separability vs per-site D performance (FedAvg MLP)")
    fig_j.update_traces(textposition="top center", marker=dict(color="#d62728"))
    fig_j.write_html(OUT_DIR / "siteD_separability_vs_FL.html",
                     include_plotlyjs=HTML_PLOTLYJS, full_html=True)
    fig_j.show()
else:
    print("No FL final_results.csv found under", FL_DIR, "- skipping join plot")

## How to use the interactive figures

**Color by** switches the grouping (Site / Species / Resistance / Cluster);
**Highlight** dims everything except the chosen group (useful: *D vs rest*,
*D only*, and the D-dominant clusters listed at the bottom). Click legend
entries to hide/show categories. Hover any point for its provenance
(`code`, `species`, `site`, drug coverage / R fraction, cluster).

In **local JupyterLab**, figures are `FigureWidget`s: use the lasso or
box-select tool to select points and the provenance table underneath updates
live. In **Colab** selection callbacks are unavailable; use the dropdowns and
hover instead.

All views are also saved as standalone HTML in the results directory
(`umap_global_interactive.html`, `umap_<drug>_interactive.html`,
`umap_siteD_interactive.html`, `site_separability.html`, ...), so they can be
shared and opened in any browser.

**Outputs**

| File | Content |
|---|---|
| `cluster_composition_global.csv` | global clusters ranked by D-lift + species/resistance composition |
| `cluster_composition_<drug>.csv` | same per drug |
| `cluster_composition_siteD.csv` | clusters *within* site D |
| `site_separability.csv` | one-vs-rest silhouette / 5-NN / linear-probe per site, per mask |
| `siteD_separability_per_drug.csv` | D-vs-rest separability per drug |
| `siteD_separability_vs_FL.csv` | join with per-drug D_BalAcc (when FL runs are found) |
| `siteD_cluster_overview.pdf` | static snapshot for docs/paper |

In [ ]:
print("=" * 70)
print(f"  Interactive UMAP diagnostic complete ({RUN_TAG})")
print(f"  Outputs in {OUT_DIR.resolve()}")
for f in sorted(OUT_DIR.glob("*")):
    if f.is_file():
        print(f"    {f.name}")
print("\nSite-D interpretation targets:")
print("  - D_only/global UMAP: which clusters are D-dominated (D-lift) and their species")
print("  - Hover / select points: provenance (code, species, site, drug coverage, R fraction)")
print("  - Highlight dropdown: 'D vs rest' and D-dominant clusters")
print("  - site_separability.csv: is D the most separable site? does masking change that?")
print("  - siteD_separability_per_drug.csv (+_vs_FL): D separability vs FL D performance")
if len(fl_df) == 0:
    print("  (no FL run dirs found locally -> join plot skipped; run in Colab/Drive for the join)")

del raw
gc.collect()